In [1]:
from pathlib import Path
import csv

run_folder = Path(
    "/scratch/project_2009497/Hungary_Data_Test/"
    "Hungary_Splits_Fidesz1_Test/run_20260920_150721_abc3f6"
)

with (run_folder / "summary.csv").open() as f:
    failed = [
        row for row in csv.DictReader(f)
        if row["status"] == "failed"
    ]

for row in failed[:2]:
    print(f"\nVIDEO: {row['source']}")
    log = Path(row["log"])
    print("\n".join(
        log.read_text(errors="replace").splitlines()[-80:]
    ))


VIDEO: /scratch/project_2009497/Hungary_Data_Test/Hungary_Organised/Fidesz/1/20260328/TikTok/az_recorder_20260328_120711.mp4
  Stream #0:0[0x1](eng): Video: h264 (High) (avc1 / 0x31637661), yuv420p(tv, bt709, progressive), 720x1600, 9004 kb/s, 29.84 fps, 90k tbr, 90k tbn (default)
    Metadata:
      creation_time   : 2026-03-28T11:26:04.000000Z
      handler_name    : VideoHandle
      vendor_id       : [0][0][0][0]
  Stream #0:1[0x2](eng): Audio: aac (LC) (mp4a / 0x6134706D), 44100 Hz, mono, fltp, 128 kb/s (default)
    Metadata:
      creation_time   : 2026-03-28T11:26:04.000000Z
      handler_name    : SoundHandle
      vendor_id       : [0][0][0][0]
Stream mapping:
  Stream #0:0 -> #0:0 (h264 (native) -> h264 (libx264))
  Stream #0:1 -> #0:1 (aac (native) -> aac (native))
Press [q] to stop, [?] for help
[libx264 @ 0x558bda542ec0] using cpu capabilities: MMX2 SSE2Fast SSSE3 SSE4.2 AVX FMA3 BMI2 AVX2 AVX512
[libx264 @ 0x558bda542ec0] profile High, level 3.2, 4:2:0, 8-bit
[libx264 @

In [2]:
from pathlib import Path
import shutil
import subprocess
import tempfile

source = Path(
    "/scratch/project_2009497/Hungary_Data_Test/Hungary_Organised/"
    "Fidesz/1/20260328/TikTok/az_recorder_20260328_120711.mp4"
)

base = Path("/scratch/project_2009497/Hungary_Data_Test/FFmpeg_Diagnostics")
base.mkdir(parents=True, exist_ok=True)
test_folder = Path(tempfile.mkdtemp(prefix="timestamp_test_", dir=base))

ffmpeg = shutil.which("ffmpeg")
if ffmpeg is None:
    raise RuntimeError("FFmpeg executable was not found.")

for label, options in [
    ("original_settings", []),
    ("timescale_90000", ["-video_track_timescale", "90000"]),
]:
    output = test_folder / f"{label}.mp4"

    command = [
        ffmpeg, "-hide_banner", "-nostdin", "-n",
        "-loglevel", "warning",
        "-threads", "1",
        "-ss", "0", "-to", "30", "-i", str(source),
        "-threads", "1",
        "-acodec", "aac", "-strict", "experimental",
        "-b:a", "192k",
        *options,
        str(output),
    ]

    print(f"\nTesting: {label}", flush=True)
    result = subprocess.run(command, capture_output=True, text=True)

    (test_folder / f"{label}.log").write_text(result.stderr)
    print("Export succeeded" if result.returncode == 0 else "Export FAILED")
    print(result.stderr[-4000:])

print("\nDiagnostic files:", test_folder)


Testing: original_settings
Export FAILED
[mp4 @ 0x563caade9400] pts/dts pair unsupported
[vost#0:0/libx264 @ 0x563caacf5080] Error submitting a packet to the muxer: Not yet implemented in FFmpeg, patches welcome
    Last message repeated 1 times
[out#0/mp4 @ 0x563caae75f00] Error muxing a packet
[out#0/mp4 @ 0x563caae75f00] Task finished with error code: -1163346256 (Not yet implemented in FFmpeg, patches welcome)
[out#0/mp4 @ 0x563caae75f00] Terminating thread with return code -1163346256 (Not yet implemented in FFmpeg, patches welcome)


Testing: timescale_90000
Export succeeded


Diagnostic files: /scratch/project_2009497/Hungary_Data_Test/FFmpeg_Diagnostics/timestamp_test_k3fs7fkv
